# 🌊 SST Amérique Centrale 2023–2024
### Copernicus Marine — OSTIA L4 REP (reprocessed)

**Dataset** : `METOFFICE-GLO-SST-L4-REP-OBS-SST`  
**Région** : Amérique Centrale + Caraïbes  
**Variable** : `analysed_sst` (Kelvin → °C)

> Compte gratuit requis : [marine.copernicus.eu](https://marine.copernicus.eu)  
> Connexion : `copernicusmarine login` dans le terminal


## 0. Installation

In [ ]:
!micromamba install -c conda-forge --yes copernicusmarine xarray matplotlib cartopy 2>/dev/null | tail -2
print("✓ OK")

## 1. Imports & paramètres

In [ ]:
import copernicusmarine
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

DATA_DIR = Path("data_sst")
DATA_DIR.mkdir(exist_ok=True)

DATASET_ID = "METOFFICE-GLO-SST-L4-REP-OBS-SST"
VARIABLE   = "analysed_sst"

LON_MIN, LON_MAX = -120, -60
LAT_MIN, LAT_MAX =    -5,  30

print(f"Région  : lon [{LON_MIN}, {LON_MAX}]  lat [{LAT_MIN}, {LAT_MAX}]")
print(f"Dataset : {DATASET_ID}")

## 2. Téléchargement
Les fichiers NetCDF sont sauvegardés dans `data_sst/`.  
Si le fichier existe déjà, le téléchargement est ignoré.

In [ ]:
for year in [2023, 2024]:
    out_file = DATA_DIR / f"sst_central_america_{year}.nc"

    if out_file.exists():
        print(f"✓ {out_file} déjà présent, ignoré.")
        continue

    print(f"⬇ Téléchargement {year}...")
    copernicusmarine.subset(
        dataset_id        = DATASET_ID,
        variables         = [VARIABLE],
        minimum_longitude = LON_MIN,
        maximum_longitude = LON_MAX,
        minimum_latitude  = LAT_MIN,
        maximum_latitude  = LAT_MAX,
        start_datetime    = f"{year}-01-01T00:00:00",
        end_datetime      = f"{year}-12-31T23:59:59",
        output_directory  = str(DATA_DIR),
        output_filename   = out_file.name,
        overwrite         = False,
    )
    print(f"✓ {out_file}  ({out_file.stat().st_size / 1e6:.1f} Mo)")

print("\nTéléchargement terminé.")

## 3. Carte SST moyenne annuelle

In [ ]:
for year in [2023, 2024]:
    nc = DATA_DIR / f"sst_central_america_{year}.nc"
    if not nc.exists():
        print(f"⚠ {nc} introuvable, ignoré.")
        continue

    ds  = xr.open_dataset(nc)
    sst = ds[VARIABLE].mean("time") - 273.15  # Kelvin → Celsius

    fig, ax = plt.subplots(figsize=(10, 6),
                           subplot_kw={"projection": ccrs.PlateCarree()})
    sst.plot(ax=ax, transform=ccrs.PlateCarree(),
             cmap="RdYlBu_r", vmin=24, vmax=32,
             cbar_kwargs={"label": "SST (°C)"})
    ax.add_feature(cfeature.LAND,      facecolor="#d4c9a8", zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7,       zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle="--", zorder=3)
    ax.gridlines(draw_labels=True, linewidth=0.3, linestyle="--", color="gray")
    ax.set_title(f"SST moyenne annuelle {year} — Amérique Centrale", fontsize=13)
    fig.tight_layout()
    out = DATA_DIR / f"sst_map_{year}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Carte sauvegardée : {out}")